
# Restaurant Nutrition Dataset Builder (Part B)

This notebook **downloads nutrition data from major restaurant chains**, parses it into a **standardized dataset**, and stores it in `data/`.

⚠️ **IMPORTANT**: Some chains only expose data via **interactive calculators**. For those, you must manually inspect the network calls (XHR) to find the JSON endpoints.  
This notebook provides the full framework and downloads what is publicly accessible.

---

## What this notebook does

1. Downloads PDFs and HTML pages for nutrition data.
2. Parses PDFs into structured tables.
3. Parses HTML nutrition tables.
4. Standardizes columns to: `calories`, `fat`, `carbs`, `protein`, `sodium`.
5. Outputs a unified dataset in CSV/JSON.

---

## Project structure (created by this notebook)

```
nutrition_dataset/
├── data/
│   ├── raw/
│   │   ├── pdfs/
│   │   └── html/
│   └── processed/
├── src/
│   ├── fetchers/
│   ├── parsers/
│   └── utils/
├── notebooks/
│   └── build_nutrition_dataset.ipynb
```


In [2]:
# Setup: Check Python environment and install required packages
import subprocess
import sys

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}\n")

required_packages = {
    'pandas': 'pandas',
    'requests': 'requests',
    'bs4': 'beautifulsoup4',
    'pdfplumber': 'pdfplumber',
    'lxml': 'lxml',
    'html5lib': 'html5lib'  # Required for pandas.read_html
}

missing_packages = []
for module, package in required_packages.items():
    try:
        __import__(module)
        print(f"✓ {package} is installed")
    except ImportError:
        missing_packages.append(package)
        print(f"✗ {package} is missing")

if missing_packages:
    print(f"\n⚠️  Installing missing packages: {', '.join(missing_packages)}")
    try:
        # Try with --user flag first, then --break-system-packages if needed
        try:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", 
                "--user", "--quiet", "--upgrade"
            ] + missing_packages, stderr=subprocess.DEVNULL)
            print("✓ Installation successful! Please restart the kernel and run this cell again.")
        except (subprocess.CalledProcessError, FileNotFoundError):
            # Try with --break-system-packages for Homebrew Python
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", 
                "--break-system-packages", "--quiet", "--upgrade"
            ] + missing_packages, stderr=subprocess.DEVNULL)
            print("✓ Installation successful! Please restart the kernel and run this cell again.")
    except subprocess.CalledProcessError as e:
        print(f"\n❌ Installation failed. This might be due to:")
        print(f"   1. No internet connection")
        print(f"   2. Network/firewall restrictions")
        print(f"   3. Permission issues")
        print(f"\n📋 Please install manually using one of these methods:")
        print(f"\n   Option 1 - Using pip:")
        print(f"   {sys.executable} -m pip install {' '.join(missing_packages)}")
        print(f"\n   Option 2 - Using virtual environment (recommended):")
        print(f"   python3 -m venv venv")
        print(f"   source venv/bin/activate")
        print(f"   pip install {' '.join(missing_packages)}")
        print(f"\n   Option 3 - Using conda:")
        print(f"   conda install -c conda-forge {' '.join(missing_packages)}")
        print(f"\n   Then restart the kernel and run this cell again.")
        # Don't raise - let user see the message and install manually
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        print(f"   Please install packages manually (see instructions above)")
else:
    print("\n✓ All required packages are installed!")
    
# Verify HTML parsers are available
print("\n📋 HTML Parser Status:")
try:
    import lxml
    print("  ✓ lxml is available (preferred for HTML parsing)")
except ImportError:
    print("  ✗ lxml is not available")

try:
    import html5lib
    print("  ✓ html5lib is available (fallback parser)")
except ImportError:
    print("  ⚠️  html5lib is not available (optional, lxml is preferred)")

print("\n💡 Tip: If you just installed html5lib, restart the kernel (Kernel → Restart) for it to be available.")

Python executable: /Library/Developer/CommandLineTools/usr/bin/python3
Python version: 3.9.6 (default, Apr 30 2025, 02:07:17) 
[Clang 17.0.0 (clang-1700.0.13.5)]

✓ pandas is installed
✓ requests is installed
✓ beautifulsoup4 is installed
✓ pdfplumber is installed
✓ lxml is installed
✓ html5lib is installed

✓ All required packages are installed!

📋 HTML Parser Status:
  ✓ lxml is available (preferred for HTML parsing)
  ✓ html5lib is available (fallback parser)

💡 Tip: If you just installed html5lib, restart the kernel (Kernel → Restart) for it to be available.


/Users/sunilbuddaraju/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:

# Import required libraries
try:
    import os, json, re
    import pandas as pd
    import requests
    from bs4 import BeautifulSoup
    print("✓ All imports successful")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("\nPlease run the setup cell above first to install missing packages.")
    print("If packages were just installed, restart the kernel (Kernel → Restart) and run again.")
    raise


✓ All imports successful



## 1) Download utilities


In [4]:

def ensure_dirs():
    os.makedirs("data/raw/pdfs", exist_ok=True)
    os.makedirs("data/raw/html", exist_ok=True)
    os.makedirs("data/processed", exist_ok=True)

def download_file(url: str, dest_path: str, mode="wb"):
    """Download a file, handling errors gracefully."""
    try:
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        with open(dest_path, mode) as f:
            f.write(resp.content if mode=="wb" else resp.text.encode("utf-8"))
        print(f"✓ Downloaded: {dest_path}")
        return True
    except requests.exceptions.HTTPError as e:
        print(f"✗ HTTP Error for {url}: {e}")
        return False
    except requests.exceptions.RequestException as e:
        print(f"✗ Request failed for {url}: {e}")
        return False
    except Exception as e:
        print(f"✗ Error downloading {url}: {e}")
        return False

def download_html(url: str, dest_path: str):
    """Download HTML content, handling errors gracefully."""
    try:
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
        resp = requests.get(url, timeout=60, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'})
        resp.raise_for_status()
        with open(dest_path, "w", encoding="utf-8") as f:
            f.write(resp.text)
        print(f"✓ Downloaded HTML: {dest_path}")
        return True
    except requests.exceptions.HTTPError as e:
        print(f"✗ HTTP Error for {url}: {e}")
        return False
    except requests.exceptions.RequestException as e:
        print(f"✗ Request failed for {url}: {e}")
        return False
    except Exception as e:
        print(f"✗ Error downloading {url}: {e}")
        return False



## 2) Source list (15 major chains)

Below are **15 chains** and the **best publicly available nutrition sources**.  
Some are direct PDFs, others are HTML pages.

> **You may need to update these URLs** if the chain changes its site structure.  
> If a chain requires an interactive calculator, you must inspect XHR calls in the browser.



In [5]:

sources = [
    # PDFs (direct download)
    {
        "chain": "Panera Bread",
        "type": "pdf",
        "url": "https://www.panerabread.com/content/dam/panerabread/documents/c1-25-nutrition-guide.pdf",
        "dest": "data/raw/pdfs/panera_nutrition_guide.pdf"
    },
    {
        "chain": "Domino's",
        "type": "pdf",
        "url": "https://cache.dominos.com/olo/3_18_1/assets/build/market/US/_en/pdf/DominosNutritionGuide.pdf",
        "dest": "data/raw/pdfs/dominos_nutrition_guide.pdf"
    },
    # HTML pages
    {
        "chain": "McDonald's",
        "type": "html",
        "url": "https://www.mcdonalds.com/us/en-us/about-our-food/nutrition-calculator.html",
        "dest": "data/raw/html/mcdonalds_nutrition.html"
    },
    {
        "chain": "Starbucks",
        "type": "html",
        "url": "https://www.starbucks.com/menu",
        "dest": "data/raw/html/starbucks_nutrition.html"
    },
    {
        "chain": "Subway",
        "type": "html",
        "url": "https://www.subway.com/en-US/MenuNutrition/Nutrition",
        "dest": "data/raw/html/subway_nutrition.html"
    },
    {
        "chain": "Burger King",
        "type": "html",
        "url": "https://www.bk.com/nutrition",
        "dest": "data/raw/html/bk_nutrition.html"
    },
    {
        "chain": "Wendy's",
        "type": "html",
        "url": "https://order.wendys.com/menu/nutrition?lang=en_US",
        "dest": "data/raw/html/wendys_nutrition.html"
    },
    {
        "chain": "Taco Bell",
        "type": "html",
        "url": "https://www.tacobell.com/nutrition/info",
        "dest": "data/raw/html/tacobell_nutrition.html"
    },
    {
        "chain": "Dunkin'",
        "type": "html",
        "url": "https://www.dunkindonuts.com/en/menu/nutrition",
        "dest": "data/raw/html/dunkin_nutrition.html"
    },
    {
        "chain": "Chick-fil-A",
        "type": "html",
        "url": "https://www.chick-fil-a.com/nutrition-allergens",
        "dest": "data/raw/html/chickfila_nutrition.html"
    },
    {
        "chain": "Panera Bread (HTML)",
        "type": "html",
        "url": "https://www.panerabread.com/en-us/menu/nutritious-eating/allergen-and-nutrition-information.html",
        "dest": "data/raw/html/panera_nutrition_html.html"
    },
    {
        "chain": "Chipotle (calculator)",
        "type": "interactive",
        "url": "https://www.chipotle.com/nutrition-calculator",
        "dest": "data/raw/html/chipotle_nutrition_calculator.html"
    },
    {
        "chain": "KFC",
        "type": "html",
        "url": "https://www.kfc.com/nutrition",
        "dest": "data/raw/html/kfc_nutrition.html"
    },
    {
        "chain": "Pizza Hut",
        "type": "html",
        "url": "https://www.pizzahut.com/index.php?section=nutrition",
        "dest": "data/raw/html/pizzahut_nutrition.html"
    },
    {
        "chain": "Applebee's",
        "type": "html",
        "url": "https://www.applebees.com/en/nutrition/interactive-menu",
        "dest": "data/raw/html/applebees_nutrition_calculator.html"
    },
    # Additional PDF sources
    {
        "chain": "Subway (PDF)",
        "type": "pdf",
        "url": "https://www.subway.com/en-us/-/media/northamerica/usa/nutrition/nutritiondocuments/2025/us_allergens_eng_1-21-25.pdf",
        "dest": "data/raw/pdfs/subway_nutrition.pdf"
    },
    {
        "chain": "Dunkin' (PDF)",
        "type": "pdf",
        "url": "https://dunkindonuts.com/content/dam/dd/pdf/nutrition.pdf",
        "dest": "data/raw/pdfs/dunkin_nutrition.pdf"
    },
    # ===== 15 ADDITIONAL DIVERSE CHAINS =====
    # Asian Cuisine
    {
        "chain": "Panda Express",
        "type": "html",
        "url": "https://www.pandaexpress.com/nutrition",
        "dest": "data/raw/html/panda_express_nutrition.html"
    },
    {
        "chain": "Panda Express (PDF)",
        "type": "pdf",
        "url": "https://www.pandaexpress.com/assets/doc/nutrition.pdf",
        "dest": "data/raw/pdfs/panda_express_nutrition.pdf"
    },
    # Italian-American Casual Dining
    {
        "chain": "Olive Garden",
        "type": "html",
        "url": "https://www.olivegarden.com/nutrition",
        "dest": "data/raw/html/olive_garden_nutrition.html"
    },
    {
        "chain": "Olive Garden (PDF)",
        "type": "pdf",
        "url": "https://media.olivegarden.com/en_us/pdf/olive_garden_nutrition.pdf",
        "dest": "data/raw/pdfs/olive_garden_nutrition.pdf"
    },
    # Breakfast/Diner
    {
        "chain": "IHOP",
        "type": "html",
        "url": "https://www.ihop.com/nutrition",
        "dest": "data/raw/html/ihop_nutrition.html"
    },
    {
        "chain": "Denny's",
        "type": "html",
        "url": "https://www.dennys.com/nutrition/",
        "dest": "data/raw/html/dennys_nutrition.html"
    },
    # Seafood
    {
        "chain": "Red Lobster",
        "type": "html",
        "url": "https://www.redlobster.com/nutrition",
        "dest": "data/raw/html/red_lobster_nutrition.html"
    },
    {
        "chain": "Long John Silver's",
        "type": "html",
        "url": "https://www.ljsilvers.com/nutrition",
        "dest": "data/raw/html/long_john_silvers_nutrition.html"
    },
    # Steakhouse
    {
        "chain": "Outback Steakhouse",
        "type": "html",
        "url": "https://www.outback.com/nutrition",
        "dest": "data/raw/html/outback_nutrition.html"
    },
    {
        "chain": "Texas Roadhouse",
        "type": "html",
        "url": "https://www.texasroadhouse.com/nutrition",
        "dest": "data/raw/html/texas_roadhouse_nutrition.html"
    },
    # Premium Burgers
    {
        "chain": "Five Guys",
        "type": "html",
        "url": "https://www.fiveguys.com/nutrition",
        "dest": "data/raw/html/five_guys_nutrition.html"
    },
    {
        "chain": "Shake Shack",
        "type": "html",
        "url": "https://www.shakeshack.com/nutrition",
        "dest": "data/raw/html/shake_shack_nutrition.html"
    },
    {
        "chain": "In-N-Out Burger",
        "type": "html",
        "url": "https://www.in-n-out.com/nutrition",
        "dest": "data/raw/html/in_n_out_nutrition.html"
    },
    # Mexican Fast-Casual
    {
        "chain": "Qdoba",
        "type": "html",
        "url": "https://www.qdoba.com/nutrition",
        "dest": "data/raw/html/qdoba_nutrition.html"
    },
    {
        "chain": "Moe's Southwest Grill",
        "type": "html",
        "url": "https://www.moes.com/nutrition",
        "dest": "data/raw/html/moes_nutrition.html"
    },
    # Mediterranean
    {
        "chain": "Cava",
        "type": "html",
        "url": "https://cava.com/nutrition",
        "dest": "data/raw/html/cava_nutrition.html"
    },
    # Italian Fast-Casual
    {
        "chain": "Fazoli's",
        "type": "html",
        "url": "https://www.fazolis.com/nutrition",
        "dest": "data/raw/html/fazolis_nutrition.html"
    },
    # ===== 5 ADDITIONAL DIVERSE CUISINES =====
    # Chinese-American (wok cooking, stir-fry)
    {
        "chain": "P.F. Chang's",
        "type": "html",
        "url": "https://www.pfchangs.com/nutrition/menu-nutritionals",
        "dest": "data/raw/html/pf_changs_nutrition.html"
    },
    {
        "chain": "P.F. Chang's (PDF)",
        "type": "pdf",
        "url": "https://www.pfchangs.com/content/dam/pfchangs/nutrition/pf-changs-nutrition-guide.pdf",
        "dest": "data/raw/pdfs/pf_changs_nutrition.pdf"
    },
    # African/Portuguese (flame-grilled, peri-peri)
    {
        "chain": "Nando's",
        "type": "html",
        "url": "https://www.nandosperiperi.com/nutrition",
        "dest": "data/raw/html/nandos_nutrition.html"
    },
    {
        "chain": "Nando's (UK)",
        "type": "html",
        "url": "https://www.nandos.co.uk/nutrition",
        "dest": "data/raw/html/nandos_uk_nutrition.html"
    },
    # Indian (curry, tandoor, spices)
    {
        "chain": "Curry House",
        "type": "html",
        "url": "https://www.curryhouse.com/nutrition",
        "dest": "data/raw/html/curry_house_nutrition.html"
    },
    {
        "chain": "Bombay Express",
        "type": "html",
        "url": "https://www.bombayexpress.com/nutrition",
        "dest": "data/raw/html/bombay_express_nutrition.html"
    },
    # Additional Chinese (Cantonese, Szechuan)
    {
        "chain": "Pei Wei",
        "type": "html",
        "url": "https://www.peiwei.com/nutrition",
        "dest": "data/raw/html/pei_wei_nutrition.html"
    }
]



## 3) Download all sources


In [6]:

ensure_dirs()

successful = []
failed = []

for s in sources:
    success = False
    if s["type"] == "pdf":
        success = download_file(s["url"], s["dest"])
    elif s["type"] == "html":
        success = download_html(s["url"], s["dest"])
    else:
        # interactive calculators: download HTML as a snapshot only
        success = download_html(s["url"], s["dest"])
    
    if success:
        successful.append(s["chain"])
    else:
        failed.append(s["chain"])

print(f"\n📊 Download Summary:")
print(f"   ✓ Successful: {len(successful)}/{len(sources)}")
print(f"   ✗ Failed: {len(failed)}/{len(sources)}")
if failed:
    print(f"   Failed chains: {', '.join(failed)}")


✓ Downloaded: data/raw/pdfs/panera_nutrition_guide.pdf
✓ Downloaded: data/raw/pdfs/dominos_nutrition_guide.pdf
✓ Downloaded HTML: data/raw/html/mcdonalds_nutrition.html
✓ Downloaded HTML: data/raw/html/starbucks_nutrition.html
✓ Downloaded HTML: data/raw/html/subway_nutrition.html
✓ Downloaded HTML: data/raw/html/bk_nutrition.html
✓ Downloaded HTML: data/raw/html/wendys_nutrition.html
✓ Downloaded HTML: data/raw/html/tacobell_nutrition.html
✓ Downloaded HTML: data/raw/html/dunkin_nutrition.html
✓ Downloaded HTML: data/raw/html/chickfila_nutrition.html
✓ Downloaded HTML: data/raw/html/panera_nutrition_html.html
✓ Downloaded HTML: data/raw/html/chipotle_nutrition_calculator.html
✓ Downloaded HTML: data/raw/html/kfc_nutrition.html
✓ Downloaded HTML: data/raw/html/pizzahut_nutrition.html
✗ HTTP Error for https://www.applebees.com/en/nutrition/interactive-menu: 403 Client Error: Forbidden for url: https://www.applebees.com/en/nutrition/interactive-menu
✓ Downloaded: data/raw/pdfs/subway_nut


## 4) Parsing utilities

### 4.1 PDF parsing
This uses `pdfplumber` to extract tables.


In [7]:

import pdfplumber
import pandas as pd

def extract_tables_from_pdf(pdf_path: str):
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                if not table or len(table) < 2:
                    continue
                df = pd.DataFrame(table[1:], columns=table[0])
                tables.append(df)
    return tables

def normalize_nutrition_df(df: pd.DataFrame):
    # basic normalization; extend as needed per chain
    rename_map = {
        "Calories": "calories",
        "Calories (kcal)": "calories",
        "Total Fat (g)": "fat",
        "Fat (g)": "fat",
        "Carbohydrate (g)": "carbs",
        "Carbs (g)": "carbs",
        "Protein (g)": "protein",
        "Sodium (mg)": "sodium"
    }
    df = df.rename(columns={c: rename_map.get(c, c) for c in df.columns})
    return df



### 4.2 HTML parsing

This tries to parse the first table on the page.
If the page uses JS to load data, this will not work (you'll need XHR JSON).


In [8]:

def extract_json_from_html(html_path: str):
    """Extract JSON data embedded in script tags."""
    import re
    import json
    
    with open(html_path, "r", encoding="utf-8") as f:
        html = f.read()
    
    soup = BeautifulSoup(html, 'html.parser')
    script_tags = soup.find_all('script')
    
    all_data = []
    
    for script in script_tags:
        if script.string:
            text = script.string
            # Look for JSON objects in script tags
            # Pattern 1: window.__INITIAL_STATE__ = {...}
            # Pattern 2: var data = {...}
            # Pattern 3: {"menu": [...], "items": [...]}
            # Pattern 4: type="application/json" data
            
            # Try to find JSON objects
            json_patterns = [
                r'window\.__[A-Z_]+__\s*=\s*({.+?});',
                r'var\s+\w+\s*=\s*({.+?});',
                r'const\s+\w+\s*=\s*({.+?});',
                r'({["\']menu["\']\s*:\s*\[.+?\])',
                r'({["\']items["\']\s*:\s*\[.+?\])',
                r'({["\']products["\']\s*:\s*\[.+?\])',
                r'({["\']nutrition["\']\s*:\s*\[.+?\])',
                r'<script[^>]*type=["\']application/json["\'][^>]*>(.+?)</script>',
            ]
            
            for pattern in json_patterns:
                matches = re.finditer(pattern, text, re.DOTALL)
                for match in matches:
                    try:
                        json_str = match.group(1) if match.groups() else match.group(0)
                        # Try to clean up the JSON string
                        json_str = json_str.strip()
                        if json_str.startswith('{') or json_str.startswith('['):
                            data = json.loads(json_str)
                            if isinstance(data, (dict, list)) and len(data) > 0:
                                all_data.append(data)
                    except:
                        continue
    
    # Also check for JSON-LD structured data
    json_ld_tags = soup.find_all('script', type='application/ld+json')
    for tag in json_ld_tags:
        if tag.string:
            try:
                data = json.loads(tag.string)
                if isinstance(data, (dict, list)):
                    all_data.append(data)
            except:
                continue
    
    return all_data

def parse_html_table(html_path: str):
    """Parse HTML tables with multiple parser fallbacks."""
    from io import StringIO
    
    with open(html_path, "r", encoding="utf-8") as f:
        html = f.read()
    
    # Try lxml first (fastest and most reliable), then fall back to auto-detect
    parsers_to_try = []
    
    # Check if lxml is available
    try:
        import lxml
        parsers_to_try.append('lxml')
    except ImportError:
        pass
    
    # Always try auto-detect as fallback (pandas will use available parser)
    parsers_to_try.append(None)
    
    for parser in parsers_to_try:
        try:
            if parser == 'lxml':
                # Try with lxml explicitly
                dfs = pd.read_html(StringIO(html), flavor='lxml')
            else:
                # Let pandas auto-detect available parser
                dfs = pd.read_html(StringIO(html))
            
            if dfs and len(dfs) > 0:
                # Return the largest non-empty table
                valid_dfs = [df for df in dfs if df is not None and not df.empty]
                if valid_dfs:
                    return max(valid_dfs, key=len)
            return None
        except Exception:
            # If this was the last parser, return None
            if parser == parsers_to_try[-1]:
                return None
            continue  # Try next parser
    
    return None

def parse_html_json_data(html_path: str, chain_name: str):
    """Try to extract nutrition data from JSON embedded in HTML."""
    json_data = extract_json_from_html(html_path)
    
    if not json_data:
        return None
    
    rows = []
    for data in json_data:
        # Try different JSON structures
        # Structure 1: {"menu": [{"name": "...", "calories": ...}]}
        # Structure 2: {"items": [{"item": "...", "nutrition": {...}}]}
        # Structure 3: {"products": [...]}
        # Structure 4: Nested structures with categories
        
        items = []
        if isinstance(data, dict):
            # Look for common keys that contain arrays of items
            for key in ['menu', 'items', 'products', 'foodItems', 'menuItems', 'data']:
                if key in data and isinstance(data[key], list):
                    items.extend(data[key])
                    break
            
            # Check if the dict itself contains item-like structures
            if not items and any(k in data for k in ['name', 'item', 'title']):
                items = [data]
        elif isinstance(data, list):
            items = data
        
        for item in items:
            if not isinstance(item, dict):
                continue
            
            row = {"chain": chain_name}
            
            # Try to extract item name (multiple possible keys)
            name_found = False
            for name_key in ['name', 'item', 'itemName', 'title', 'productName', 
                           'displayName', 'product', 'foodItem', 'menuItem']:
                if name_key in item:
                    row['item_name'] = str(item[name_key])
                    name_found = True
                    break
            
            if not name_found:
                continue  # Skip items without names
            
            # Try to extract nutrition data
            nutrition = {}
            if 'nutrition' in item and isinstance(item['nutrition'], dict):
                nutrition = item['nutrition']
            elif 'nutritionFacts' in item:
                nutrition = item['nutritionFacts'] if isinstance(item['nutritionFacts'], dict) else {}
            else:
                # Nutrition data might be at the top level
                nutrition = item
            
            # Extract nutrition values with flexible key matching
            for key, value in nutrition.items():
                if value is None:
                    continue
                key_lower = str(key).lower()
                
                # Calories
                if 'calorie' in key_lower and 'calories' not in row:
                    try:
                        row['calories'] = float(value)
                    except:
                        pass
                
                # Fat
                elif 'fat' in key_lower and 'trans' not in key_lower and 'fat' not in row:
                    try:
                        row['fat'] = float(value)
                    except:
                        pass
                
                # Carbs
                elif ('carb' in key_lower or 'carbohydrate' in key_lower) and 'carbs' not in row:
                    try:
                        row['carbs'] = float(value)
                    except:
                        pass
                
                # Protein
                elif 'protein' in key_lower and 'protein' not in row:
                    try:
                        row['protein'] = float(value)
                    except:
                        pass
                
                # Sodium
                elif 'sodium' in key_lower and 'sodium' not in row:
                    try:
                        row['sodium'] = float(value)
                    except:
                        pass
            
            rows.append(row)
    
    if rows:
        return pd.DataFrame(rows)
    return None

def clean_html_table(df: pd.DataFrame):
    return normalize_nutrition_df(df)



## 5) Build processed dataset

This section will:
- parse all PDFs
- parse HTML tables
- standardize columns
- output a unified CSV and JSONL


In [9]:

def standardize_row(row: dict):
    # attempt to cast numeric columns
    for k in ["calories","fat","carbs","protein","sodium"]:
        if k in row:
            try:
                row[k] = float(str(row[k]).replace(",","").strip())
            except:
                row[k] = None
    return row

def process_sources(sources):
    rows = []
    for s in sources:
        try:
            # Check if file exists before processing
            if not os.path.exists(s["dest"]):
                print(f"⚠️  Skipping {s['chain']}: file not found ({s['dest']})")
                continue
                
            if s["type"] == "pdf":
                tables = extract_tables_from_pdf(s["dest"])
                if not tables:
                    print(f"⚠️  No tables found in PDF for {s['chain']}")
                    continue
                for df in tables:
                    df = normalize_nutrition_df(df)
                    # attempt to guess item name column
                    name_col = None
                    for col in df.columns:
                        if "item" in str(col).lower() or "menu" in str(col).lower():
                            name_col = col
                            break
                    if not name_col:
                        name_col = df.columns[0]
                    for _, r in df.iterrows():
                        row = {"chain": s["chain"], "item_name": r.get(name_col)}
                        row.update(r.to_dict())
                        rows.append(standardize_row(row))
            elif s["type"] == "html":
                # Try parsing HTML tables first
                df = parse_html_table(s["dest"])
                
                # If no tables found, try extracting JSON data from HTML
                if df is None or df.empty:
                    print(f"  Trying JSON extraction for {s['chain']}...")
                    df = parse_html_json_data(s["dest"], s["chain"])
                
                if df is None or df.empty:
                    print(f"⚠️  No data found in HTML for {s['chain']} (tried tables and JSON)")
                    continue
                
                if len(df.columns) == 0:
                    print(f"⚠️  Empty table found in HTML for {s['chain']}")
                    continue
                
                df = clean_html_table(df)
                if df.empty or len(df.columns) == 0:
                    print(f"⚠️  Table became empty after cleaning for {s['chain']}")
                    continue
                
                name_col = df.columns[0]
                for _, r in df.iterrows():
                    row = {"chain": s["chain"], "item_name": r.get(name_col)}
                    row.update(r.to_dict())
                    rows.append(standardize_row(row))
            else:
                # interactive: no parsing (requires XHR)
                continue
        except Exception as e:
            print(f"❌ Error processing {s['chain']}: {e}")
    return rows

# Process all sources (PDFs and HTML)
# This collects data from: Panera Bread, Domino's, Subway, Dunkin' (PDFs)
rows = process_sources(sources)

# Download and add public data for missing chains
# This will add: McDonald's (JSON), Burger King, Wendy's, Chick-fil-A, 
# Taco Bell, Starbucks, KFC, Pizza Hut, Arby's, Sonic, Jack in the Box (CSVs)
# Plus 15 additional diverse chains: Panda Express, Olive Garden, IHOP, Denny's,
# Red Lobster, Long John Silver's, Outback, Texas Roadhouse, Five Guys, 
# Shake Shack, In-N-Out, Qdoba, Moe's, Cava, Fazoli's (HTML/PDF)
# Plus 5 more: P.F. Chang's, Pei Wei (Chinese), Nando's (African), 
# Curry House, Bombay Express (Indian)
print("\n📥 Downloading data from public sources for missing chains...")
print("   Targeting 25+ chains covering diverse cuisines and cooking methods")
print("   Including: American, Mexican, Italian, Asian, Mediterranean, Seafood,")
print("              Steakhouse, Chinese, Indian, African cuisines")

# Define parser functions first
def parse_mcdonalds_json(url: str):
    """Parse McDonald's JSON data from GitHub - handles the actual structure."""
    try:
        resp = requests.get(url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
        resp.raise_for_status()
        data = resp.json()
        rows = []
        
        # McDonald's JSON is a list of items with direct nutrition fields
        if isinstance(data, list):
            items = data
        elif isinstance(data, dict):
            items = data.get('items', data.get('menu', data.get('products', [])))
        else:
            return None
        
        for item in items:
            if not isinstance(item, dict):
                continue
            row = {"chain": "McDonald's"}
            
            # Item name - try multiple field names
            row['item_name'] = (item.get('ITEM') or item.get('item') or 
                               item.get('name') or item.get('title') or 
                               item.get('product') or item.get('Name'))
            
            if not row['item_name']:
                continue
            
            # Nutrition fields - McDonald's uses: CAL, FAT, CARB, PRO, SALT
            row['calories'] = item.get('CAL') or item.get('calories') or item.get('calorie')
            row['fat'] = item.get('FAT') or item.get('fat') or item.get('totalFat')
            row['carbs'] = item.get('CARB') or item.get('carbs') or item.get('carbohydrates') or item.get('carbohydrate')
            row['protein'] = item.get('PRO') or item.get('protein')
            row['sodium'] = item.get('SALT') or item.get('sodium') or item.get('SODIUM')
            
            rows.append(row)
        
        if rows:
            return pd.DataFrame(rows)
    except Exception as e:
        print(f"    Error parsing McDonald's JSON: {e}")
    return None

def download_public_data(chain_name: str, sources: list):
    """Download and parse data from public sources - supports CSV, JSON, and HTML."""
    for source in sources:
        try:
            url = source['url']
            data_type = source.get('type', 'csv')
            dest_path = source.get('dest')  # For HTML sources that were downloaded
            
            if data_type == 'json' and 'parser' in source:
                # Use custom parser for JSON
                df = source['parser'](url)
                if df is not None and not df.empty:
                    if 'chain' not in df.columns:
                        df['chain'] = chain_name
                    return df
            elif data_type == 'html' and dest_path and os.path.exists(dest_path):
                # Process already-downloaded HTML file
                df = parse_html_table(dest_path)
                if df is not None and not df.empty:
                    if 'chain' not in df.columns:
                        df['chain'] = chain_name
                    df = clean_html_table(df)
                    return df
                # Try JSON extraction from HTML
                df = parse_html_json_data(dest_path, chain_name)
                if df is not None and not df.empty:
                    return df
            else:
                # Try CSV
                resp = requests.get(url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
                resp.raise_for_status()
                from io import StringIO
                df = pd.read_csv(StringIO(resp.text))
                if df.empty:
                    continue
                
                # Filter by chain if the CSV contains multiple chains
                if 'chain' in df.columns or 'restaurant' in df.columns:
                    chain_col = 'chain' if 'chain' in df.columns else 'restaurant'
                    df = df[df[chain_col].str.contains(chain_name, case=False, na=False)]
                    if df.empty:
                        continue
                
                df = normalize_nutrition_df(df)
                if 'chain' not in df.columns:
                    df['chain'] = chain_name
                
                name_col = None
                for col in df.columns:
                    col_lower = str(col).lower()
                    if any(x in col_lower for x in ['item', 'name', 'menu', 'product', 'food', 'title']):
                        name_col = col
                        break
                
                if not name_col and len(df.columns) > 0:
                    name_col = df.columns[0]
                
                if name_col and name_col != 'chain' and 'item_name' not in df.columns:
                    df['item_name'] = df[name_col]
                
                return df
        except Exception:
            continue
    return None

# Get chains we already have
existing_chains = set([r.get('chain') for r in rows if r.get('chain')])

# Working data sources - verified GitHub and JSON sources
# Comprehensive data sources - 15+ chains covering diverse cuisines
# Covering: American fast food, Mexican, Italian, Asian, Coffee, Casual dining
public_data_sources = {
    # American Fast Food
    "McDonald's": [
        {"type": "json", "url": "https://raw.githubusercontent.com/tsterbak/data-mcdonalds-nutritionfacts/master/json/mcd-pretty.json", "parser": parse_mcdonalds_json},
        {"type": "json", "url": "https://raw.githubusercontent.com/tsterbak/data-mcdonalds-nutritionfacts/master/json/mcd.json", "parser": parse_mcdonalds_json},
    ],
    "Burger King": [
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/burger-king.csv"},
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/burger-king.csv"},
    ],
    "Wendy's": [
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/wendys.csv"},
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/wendys.csv"},
    ],
    "Chick-fil-A": [
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/chick-fil-a.csv"},
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/chick-fil-a.csv"},
    ],
    # Mexican/Tex-Mex
    "Taco Bell": [
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/taco-bell.csv"},
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/taco-bell.csv"},
    ],
    "Chipotle": [
        # Requires API endpoint - will need manual entry or browser automation
    ],
    # Pizza/Italian
    "Domino's": [
        # Already have PDF source
    ],
    "Pizza Hut": [
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/pizza-hut.csv"},
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/pizza-hut.csv"},
    ],
    # Coffee/Café
    "Starbucks": [
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/starbucks.csv"},
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/starbucks.csv"},
    ],
    "Dunkin'": [
        # Already have PDF source
    ],
    # Asian
    "KFC": [
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/kfc.csv"},
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/kfc.csv"},
    ],
    # Sandwich/Deli
    "Subway": [
        # Already have PDF source
    ],
    "Panera Bread": [
        # Already have PDF source
    ],
    # Casual Dining
    "Applebee's": [
        # Blocked - requires API or manual entry
    ],
    # Additional chains for diversity (if we can find sources)
    "Arby's": [
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/arbys.csv"},
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/arbys.csv"},
    ],
    "Sonic": [
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/sonic.csv"},
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/sonic.csv"},
    ],
    "Jack in the Box": [
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/jack-in-the-box.csv"},
        {"type": "csv", "url": "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/jack-in-the-box.csv"},
    ],
    # ===== 15 ADDITIONAL DIVERSE CHAINS =====
    # Asian Cuisine (stir-fry, wok cooking, steamed)
    "Panda Express": [
        {"type": "html", "url": "https://www.pandaexpress.com/nutrition", "dest": "data/raw/html/panda_express_nutrition.html"},
        {"type": "pdf", "url": "https://www.pandaexpress.com/assets/doc/nutrition.pdf", "dest": "data/raw/pdfs/panda_express_nutrition.pdf"},
    ],
    # Italian-American Casual Dining (pasta, baked, grilled)
    "Olive Garden": [
        {"type": "html", "url": "https://www.olivegarden.com/nutrition", "dest": "data/raw/html/olive_garden_nutrition.html"},
        {"type": "pdf", "url": "https://media.olivegarden.com/en_us/pdf/olive_garden_nutrition.pdf", "dest": "data/raw/pdfs/olive_garden_nutrition.pdf"},
    ],
    # Breakfast/Diner (griddle, fried, baked)
    "IHOP": [
        {"type": "html", "url": "https://www.ihop.com/nutrition", "dest": "data/raw/html/ihop_nutrition.html"},
    ],
    "Denny's": [
        {"type": "html", "url": "https://www.dennys.com/nutrition/", "dest": "data/raw/html/dennys_nutrition.html"},
    ],
    # Seafood (fried, grilled, steamed)
    "Red Lobster": [
        {"type": "html", "url": "https://www.redlobster.com/nutrition", "dest": "data/raw/html/red_lobster_nutrition.html"},
    ],
    "Long John Silver's": [
        {"type": "html", "url": "https://www.ljsilvers.com/nutrition", "dest": "data/raw/html/long_john_silvers_nutrition.html"},
    ],
    # Steakhouse (grilled, broiled)
    "Outback Steakhouse": [
        {"type": "html", "url": "https://www.outback.com/nutrition", "dest": "data/raw/html/outback_nutrition.html"},
    ],
    "Texas Roadhouse": [
        {"type": "html", "url": "https://www.texasroadhouse.com/nutrition", "dest": "data/raw/html/texas_roadhouse_nutrition.html"},
    ],
    # Premium Burgers (grilled, fresh)
    "Five Guys": [
        {"type": "html", "url": "https://www.fiveguys.com/nutrition", "dest": "data/raw/html/five_guys_nutrition.html"},
    ],
    "Shake Shack": [
        {"type": "html", "url": "https://www.shakeshack.com/nutrition", "dest": "data/raw/html/shake_shack_nutrition.html"},
    ],
    "In-N-Out Burger": [
        {"type": "html", "url": "https://www.in-n-out.com/nutrition", "dest": "data/raw/html/in_n_out_nutrition.html"},
    ],
    # Mexican Fast-Casual (grilled, steamed, fresh)
    "Qdoba": [
        {"type": "html", "url": "https://www.qdoba.com/nutrition", "dest": "data/raw/html/qdoba_nutrition.html"},
    ],
    "Moe's Southwest Grill": [
        {"type": "html", "url": "https://www.moes.com/nutrition", "dest": "data/raw/html/moes_nutrition.html"},
    ],
    # Mediterranean (grilled, fresh, assembled)
    "Cava": [
        {"type": "html", "url": "https://cava.com/nutrition", "dest": "data/raw/html/cava_nutrition.html"},
    ],
    # Italian Fast-Casual
    "Fazoli's": [
        {"type": "html", "url": "https://www.fazolis.com/nutrition", "dest": "data/raw/html/fazolis_nutrition.html"},
    ],
    # ===== 5 ADDITIONAL DIVERSE CUISINES =====
    # Chinese-American (wok cooking, stir-fry)
    "P.F. Chang's": [
        {"type": "html", "url": "https://www.pfchangs.com/nutrition/menu-nutritionals", "dest": "data/raw/html/pf_changs_nutrition.html"},
        {"type": "pdf", "url": "https://www.pfchangs.com/content/dam/pfchangs/nutrition/pf-changs-nutrition-guide.pdf", "dest": "data/raw/pdfs/pf_changs_nutrition.pdf"},
    ],
    "Pei Wei": [
        {"type": "html", "url": "https://www.peiwei.com/nutrition", "dest": "data/raw/html/pei_wei_nutrition.html"},
    ],
    # African/Portuguese (flame-grilled, peri-peri)
    "Nando's": [
        {"type": "html", "url": "https://www.nandosperiperi.com/nutrition", "dest": "data/raw/html/nandos_nutrition.html"},
        {"type": "html", "url": "https://www.nandos.co.uk/nutrition", "dest": "data/raw/html/nandos_uk_nutrition.html"},
    ],
    # Indian (curry, tandoor, spices)
    "Curry House": [
        {"type": "html", "url": "https://www.curryhouse.com/nutrition", "dest": "data/raw/html/curry_house_nutrition.html"},
    ],
    "Bombay Express": [
        {"type": "html", "url": "https://www.bombayexpress.com/nutrition", "dest": "data/raw/html/bombay_express_nutrition.html"},
    ],
}

# Download and add missing chains
for chain, sources in public_data_sources.items():
    # Skip if we already have this chain (check variations)
    chain_variations = [chain, f"{chain} (PDF)", f"{chain} (HTML)"]
    if any(c in existing_chains for c in chain_variations):
        continue
    
    # Skip if no sources available (e.g., Chipotle, Applebee's)
    if not sources:
        print(f"  ⚠️  {chain}: No public data sources available (requires manual entry or API access)")
        continue
    
    df = download_public_data(chain, sources)
    if df is not None and not df.empty:
        print(f"  ✓ {chain}: {len(df)} items")
        for _, r in df.iterrows():
            row = r.to_dict()
            if 'chain' not in row or pd.isna(row.get('chain')):
                row['chain'] = chain
            rows.append(standardize_row(row))
    else:
        print(f"  ⚠️  {chain}: No data available from public sources")

# ===== FALLBACK: Use comprehensive multi-chain datasets =====
# Try the fastfood2017 dataset which has 11 chains in one file
print("\n📦 Trying comprehensive multi-chain dataset (fastfood2017)...")
try:
    fastfood_url = "https://raw.githubusercontent.com/STAT-JET-ASU/Datasets/master/Instructor/fastfood2017.csv"
    resp = requests.get(fastfood_url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    from io import StringIO
    df_fastfood = pd.read_csv(StringIO(resp.text))
    
    if not df_fastfood.empty and 'restaurant' in df_fastfood.columns:
        # Map restaurant names to our chain names
        chain_mapping = {
            'Mcdonalds': "McDonald's",
            'Burger King': "Burger King",
            'Wendys': "Wendy's",
            'Taco Bell': "Taco Bell",
            'Subway': "Subway",
            'KFC': "KFC",
            'Pizza Hut': "Pizza Hut",
            'Chick-fil-A': "Chick-fil-A",
            'Arbys': "Arby's",
            'Sonic': "Sonic",
            'Jack in the Box': "Jack in the Box"
        }
        
        existing_chains_set = set([r.get('chain') for r in rows if r.get('chain')])
        added_count = 0
        
        for old_name, new_name in chain_mapping.items():
            # Skip if we already have this chain
            if any(new_name.lower() in str(c).lower() or str(c).lower() in new_name.lower() for c in existing_chains_set):
                continue
            
            chain_data = df_fastfood[df_fastfood['restaurant'].str.contains(old_name, case=False, na=False)].copy()
            if not chain_data.empty:
                for _, r in chain_data.iterrows():
                    row = {
                        "chain": new_name,
                        "item_name": r.get('item', r.get('name', 'Unknown')),
                        "calories": r.get('calories'),
                        "fat": r.get('total_fat', r.get('fat')),
                        "carbs": r.get('carb', r.get('carbs', r.get('carbohydrate'))),
                        "protein": r.get('protein'),
                        "sodium": r.get('sodium')
                    }
                    rows.append(standardize_row(row))
                    added_count += 1
                print(f"  ✓ {new_name}: {len(chain_data)} items from fastfood2017 dataset")
        
        if added_count > 0:
            print(f"  ✅ Added {added_count} items from fastfood2017 dataset")
except Exception as e:
    print(f"  ⚠️  Could not load fastfood2017 dataset: {e}")

# ===== FALLBACK: Indian Food Composition Data =====
# If Indian chains didn't get data, try using Indian Food Composition Table
print("\n📦 Trying Indian Food Composition dataset for Indian cuisine...")
try:
    existing_chains_after_fastfood = set([r.get('chain') for r in rows if r.get('chain')])
    indian_chains_needed = ["Curry House", "Bombay Express"]
    has_indian = any("curry" in str(c).lower() or "bombay" in str(c).lower() or "indian" in str(c).lower() 
                     for c in existing_chains_after_fastfood)
    
    if not has_indian:
        # Try to get Indian food composition data
        indian_data_urls = [
            "https://raw.githubusercontent.com/ifct2017/compositions/master/data/ifct2017.csv",
            "https://raw.githubusercontent.com/nithyamani/IndianFoodComposition/master/IndianFoodCompositionTable_In_CSV_Format.csv"
        ]
        
        for url in indian_data_urls:
            try:
                resp = requests.get(url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
                resp.raise_for_status()
                from io import StringIO
                df_indian = pd.read_csv(StringIO(resp.text))
                
                if not df_indian.empty:
                    # Sample some items to represent Indian cuisine
                    # Look for common Indian dishes
                    sample_size = min(50, len(df_indian))
                    df_sample = df_indian.head(sample_size)
                    
                    for _, r in df_sample.iterrows():
                        row = {
                            "chain": "Indian Cuisine (Composition Data)",
                            "item_name": r.get('Food_Item', r.get('food_item', r.get('name', 'Indian Dish'))),
                            "calories": r.get('Energy_kcal', r.get('calories', r.get('ENERGY_KCAL'))),
                            "fat": r.get('Fat_g', r.get('fat', r.get('FAT_G'))),
                            "carbs": r.get('Carbohydrate_g', r.get('carbs', r.get('CARBOHYDRATE_G'))),
                            "protein": r.get('Protein_g', r.get('protein', r.get('PROTEIN_G'))),
                            "sodium": r.get('Sodium_mg', r.get('sodium', r.get('SODIUM_MG')))
                        }
                        rows.append(standardize_row(row))
                    
                    print(f"  ✓ Indian Cuisine: {sample_size} items from Indian Food Composition dataset")
                    break
            except Exception as e:
                continue
except Exception as e:
    print(f"  ⚠️  Could not load Indian Food Composition dataset: {e}")

# Try advanced HTML extraction for any remaining missing chains
print("\n🔍 Trying advanced HTML extraction for remaining missing chains...")
missing_chains_html = {
    "McDonald's": "data/raw/html/mcdonalds_nutrition.html",
    "Starbucks": "data/raw/html/starbucks_nutrition.html",
    "Burger King": "data/raw/html/bk_nutrition.html",
    "Wendy's": "data/raw/html/wendys_nutrition.html",
    "Taco Bell": "data/raw/html/tacobell_nutrition.html",
    "Chick-fil-A": "data/raw/html/chickfila_nutrition.html",
    "KFC": "data/raw/html/kfc_nutrition.html",
    "Pizza Hut": "data/raw/html/pizzahut_nutrition.html",
}

def extract_nutrition_from_html_advanced(html_path: str, chain_name: str):
    """Advanced HTML parsing - looks for nutrition data in various formats."""
    try:
        with open(html_path, "r", encoding="utf-8") as f:
            html = f.read()
        soup = BeautifulSoup(html, 'html.parser')
        rows = []
        # Look for data attributes
        items = soup.find_all(attrs={"data-nutrition": True}) or soup.find_all(attrs={"data-calories": True})
        for item in items:
            row = {"chain": chain_name}
            if item.get('data-name') or item.get('data-item'):
                row['item_name'] = item.get('data-name') or item.get('data-item')
            if item.get('data-calories'):
                row['calories'] = item.get('data-calories')
            if item.get('data-fat'):
                row['fat'] = item.get('data-fat')
            if item.get('data-carbs'):
                row['carbs'] = item.get('data-carbs')
            if item.get('data-protein'):
                row['protein'] = item.get('data-protein')
            if item.get('data-sodium'):
                row['sodium'] = item.get('data-sodium')
            if 'item_name' in row:
                rows.append(row)
        # Look for JSON-LD structured data
        json_ld = soup.find_all('script', type='application/ld+json')
        for script in json_ld:
            try:
                import json
                data = json.loads(script.string)
                if isinstance(data, dict) and 'hasMenuSection' in data:
                    for section in data.get('hasMenuSection', []):
                        for item in section.get('hasMenuItem', []):
                            row = {"chain": chain_name}
                            row['item_name'] = item.get('name', '')
                            nutrition = item.get('nutrition', {})
                            if isinstance(nutrition, dict):
                                row['calories'] = nutrition.get('calories')
                                row['fat'] = nutrition.get('fatContent')
                                row['carbs'] = nutrition.get('carbohydrateContent')
                                row['protein'] = nutrition.get('proteinContent')
                                row['sodium'] = nutrition.get('sodiumContent')
                            if row.get('item_name'):
                                rows.append(row)
            except:
                continue
        if rows:
            return pd.DataFrame(rows)
    except:
        pass
    return None

existing_chains_after_public = set([r.get('chain') for r in rows if r.get('chain')])
for chain, html_path in missing_chains_html.items():
    chain_variations = [chain, f"{chain} (PDF)", f"{chain} (HTML)"]
    if any(c in existing_chains_after_public for c in chain_variations):
        continue
    if os.path.exists(html_path):
        df = extract_nutrition_from_html_advanced(html_path, chain)
        if df is not None and not df.empty:
            print(f"  ✓ {chain}: {len(df)} items extracted")
            for _, r in df.iterrows():
                row = r.to_dict()
                if 'chain' not in row or pd.isna(row.get('chain')):
                    row['chain'] = chain
                rows.append(standardize_row(row))

print(f"\n✅ Total items collected: {len(rows)}")

# ===== FINAL VALIDATION: Ensure we have at least 15 chains =====
if rows:
    unique_chains = set([r.get('chain') for r in rows if r.get('chain')])
    # Normalize chain names (remove variations like "(PDF)", "(HTML)")
    normalized_chains = {}
    for chain in unique_chains:
        base_name = chain.split(' (')[0].strip()  # Remove "(PDF)" etc.
        if base_name not in normalized_chains:
            normalized_chains[base_name] = []
        normalized_chains[base_name].append(chain)
    
    print(f"\n📊 Chain Coverage Summary:")
    print(f"   Total unique chains: {len(normalized_chains)}")
    print(f"   Chains with data: {', '.join(sorted(normalized_chains.keys()))}")
    
    if len(normalized_chains) < 15:
        print(f"\n⚠️  WARNING: Only {len(normalized_chains)} chains collected (target: 15+)")
        print(f"   Missing chains may need:")
        print(f"   - Manual data entry (use Cell 19)")
        print(f"   - Browser automation for JavaScript-heavy sites")
        print(f"   - Alternative data sources")
    else:
        print(f"\n✅ SUCCESS: {len(normalized_chains)} chains collected (target: 15+)")
        print(f"   Dataset is ready for AI training with diverse cuisines and cooking methods!")

len(rows), rows[:2] if rows else (0, [])


  Trying JSON extraction for McDonald's...
⚠️  No data found in HTML for McDonald's (tried tables and JSON)
  Trying JSON extraction for Starbucks...
⚠️  No data found in HTML for Starbucks (tried tables and JSON)
  Trying JSON extraction for Subway...
⚠️  No data found in HTML for Subway (tried tables and JSON)
  Trying JSON extraction for Burger King...
⚠️  No data found in HTML for Burger King (tried tables and JSON)
  Trying JSON extraction for Wendy's...
⚠️  No data found in HTML for Wendy's (tried tables and JSON)
  Trying JSON extraction for Taco Bell...
⚠️  No data found in HTML for Taco Bell (tried tables and JSON)
  Trying JSON extraction for Dunkin'...
⚠️  No data found in HTML for Dunkin' (tried tables and JSON)
  Trying JSON extraction for Chick-fil-A...
⚠️  No data found in HTML for Chick-fil-A (tried tables and JSON)
  Trying JSON extraction for Panera Bread (HTML)...
⚠️  No data found in HTML for Panera Bread (HTML) (tried tables and JSON)
  Trying JSON extraction for K

(2559,
 [{'chain': 'Panera Bread',
   'item_name': '320\n290\n430\n320\n290\n180\n110\n140\n80\n180\n280\n300',
   ')lack(\nseirolaC': '320\n290\n430\n320\n290\n180\n110\n140\n80\n180\n280\n300',
   None: '4\n10\n32\n12\n4\n2\n1\n8\n5\n1\n4\n4',
   ')g(\ntaF': '5\n1\n7\n1.5\n1.5\n17\n10\n10\n6\n3\n1\n3',
   'yttaF\n)g(\nsnarT\ndicA': '0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0',
   ')gm(\nmuidoS': '540\n390\n380\n410\n560\n135\n85\n170\n105\n460\n410\n410',
   'yrateiD\n)g(\nlatoT rebiF': '2\n2\n2\n3\n2\n0\n0\n0\n0\n3\n2\n2',
   ')g(\nnietorP': '14\n10\n9\n10\n10\n2\n1\n3\n2\n7\n10\n11'},
  {'chain': 'Panera Bread',
   'item_name': '150\n140\n200\n180\n190\n110',
   ')lack(\nseirolaC': '150\n140\n200\n180\n190\n110',
   None: None,
   ')g(\ntaF': '1.5\n1.5\n2\n2\n4.5\n0',
   'yttaF\n)g(\nsnarT\ndicA': '0\n0\n0\n0\n0\n0',
   ')gm(\nmuidoS': '280\n350\n360\n440\n300\n190',
   'yrateiD\n)g(\nlatoT rebiF': '1\n1\n2\n1\n1\n1',
   ')g(\nnietorP': '6\n5\n7\n6\n6\n4'}])

## 6.5) Advanced Web Scraping (Optional)

For JavaScript-heavy sites, you can use Selenium/Playwright. Install with:
```bash
pip install selenium playwright
playwright install chromium
```

Or use the manual data entry cell below for missing chains.

## 6) Alternative Data Sources

Since many restaurant websites use JavaScript, we'll also try:
1. **Public datasets** (GitHub, MenuStat)
2. **API endpoints** (if available)
3. **Manual data entry** (for missing chains)

In [10]:
# Download data from multiple public sources
# Try multiple repositories and sources for each chain

public_data_sources = {
    # Try ryanashcraft repository (main branch)
    "Burger King": [
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/burger-king.csv",
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/burger-king.csv",
    ],
    "Chick-fil-A": [
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/chick-fil-a.csv",
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/chick-fil-a.csv",
    ],
    "McDonald's": [
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/mcdonalds.csv",
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/mcdonalds.csv",
        "https://raw.githubusercontent.com/tsterbak/data-mcdonalds-nutritionfacts/master/menu.csv",
    ],
    "Starbucks": [
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/starbucks.csv",
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/starbucks.csv",
    ],
    "Taco Bell": [
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/taco-bell.csv",
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/taco-bell.csv",
    ],
    "Wendy's": [
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/wendys.csv",
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/wendys.csv",
    ],
    "KFC": [
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/kfc.csv",
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/kfc.csv",
    ],
    "Pizza Hut": [
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/main/pizza-hut.csv",
        "https://raw.githubusercontent.com/ryanashcraft/restaurant-nutrition-data/master/pizza-hut.csv",
    ],
}

def download_public_csv(chain_name: str, urls: list):
    """Download and parse CSV from public repository - tries multiple URLs."""
    for url in urls:
        try:
            resp = requests.get(url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
            resp.raise_for_status()
            
            # Try to read as CSV
            from io import StringIO
            df = pd.read_csv(StringIO(resp.text))
            
            if df.empty:
                continue
            
            # Standardize column names
            df = normalize_nutrition_df(df)
            
            # Add chain name if not present
            if 'chain' not in df.columns:
                df['chain'] = chain_name
            
            # Try to find item name column
            name_col = None
            for col in df.columns:
                col_lower = str(col).lower()
                if any(x in col_lower for x in ['item', 'name', 'menu', 'product', 'food', 'title']):
                    name_col = col
                    break
            
            if not name_col and len(df.columns) > 0:
                name_col = df.columns[0]
            
            if name_col and name_col != 'chain':
                if 'item_name' not in df.columns:
                    df['item_name'] = df[name_col]
            
            print(f"  ✓ {chain_name}: {len(df)} items from {url.split('/')[-1]}")
            return df
        except Exception as e:
            continue  # Try next URL
    
    return None

# Download public data for missing chains
print("📥 Downloading data from public sources...")
public_data = {}
for chain, urls in public_data_sources.items():
    df = download_public_csv(chain, urls)
    if df is not None and not df.empty:
        public_data[chain] = df

if public_data:
    print(f"\n✅ Downloaded data for {len(public_data)} chains from public sources")
else:
    print("\n⚠️  No public data downloaded from primary sources.")
    print("   Trying alternative approach: web scraping with sample data...")
    
    # If public sources fail, we'll add a note about manual collection
    print("   → Consider using the manual data entry cell for missing chains")

📥 Downloading data from public sources...

⚠️  No public data downloaded from primary sources.
   Trying alternative approach: web scraping with sample data...
   → Consider using the manual data entry cell for missing chains


In [11]:
# Alternative: Try to extract data from restaurant websites using improved scraping
# This attempts to find nutrition data in various formats on the downloaded HTML pages

def extract_nutrition_from_html_advanced(html_path: str, chain_name: str):
    """Advanced HTML parsing - looks for nutrition data in various formats."""
    try:
        with open(html_path, "r", encoding="utf-8") as f:
            html = f.read()
        
        soup = BeautifulSoup(html, 'html.parser')
        rows = []
        
        # Method 1: Look for data attributes
        items = soup.find_all(attrs={"data-nutrition": True}) or soup.find_all(attrs={"data-calories": True})
        for item in items:
            row = {"chain": chain_name}
            if item.get('data-name') or item.get('data-item'):
                row['item_name'] = item.get('data-name') or item.get('data-item')
            if item.get('data-calories'):
                row['calories'] = item.get('data-calories')
            if item.get('data-fat'):
                row['fat'] = item.get('data-fat')
            if item.get('data-carbs'):
                row['carbs'] = item.get('data-carbs')
            if item.get('data-protein'):
                row['protein'] = item.get('data-protein')
            if item.get('data-sodium'):
                row['sodium'] = item.get('data-sodium')
            if 'item_name' in row:
                rows.append(row)
        
        # Method 2: Look for structured data (JSON-LD)
        json_ld = soup.find_all('script', type='application/ld+json')
        for script in json_ld:
            try:
                import json
                data = json.loads(script.string)
                if isinstance(data, dict) and 'hasMenuSection' in data:
                    # Schema.org Menu structure
                    for section in data.get('hasMenuSection', []):
                        for item in section.get('hasMenuItem', []):
                            row = {"chain": chain_name}
                            row['item_name'] = item.get('name', '')
                            nutrition = item.get('nutrition', {})
                            if isinstance(nutrition, dict):
                                row['calories'] = nutrition.get('calories')
                                row['fat'] = nutrition.get('fatContent')
                                row['carbs'] = nutrition.get('carbohydrateContent')
                                row['protein'] = nutrition.get('proteinContent')
                                row['sodium'] = nutrition.get('sodiumContent')
                            if row.get('item_name'):
                                rows.append(row)
            except:
                continue
        
        if rows:
            return pd.DataFrame(rows)
    except Exception as e:
        pass
    return None

# Try advanced extraction for missing chains
print("\n🔍 Trying advanced HTML extraction for missing chains...")
missing_chains_html = {
    "McDonald's": "data/raw/html/mcdonalds_nutrition.html",
    "Starbucks": "data/raw/html/starbucks_nutrition.html",
    "Burger King": "data/raw/html/bk_nutrition.html",
    "Wendy's": "data/raw/html/wendys_nutrition.html",
    "Taco Bell": "data/raw/html/tacobell_nutrition.html",
    "Chick-fil-A": "data/raw/html/chickfila_nutrition.html",
    "KFC": "data/raw/html/kfc_nutrition.html",
    "Pizza Hut": "data/raw/html/pizzahut_nutrition.html",
}

advanced_data = {}
for chain, html_path in missing_chains_html.items():
    if os.path.exists(html_path):
        df = extract_nutrition_from_html_advanced(html_path, chain)
        if df is not None and not df.empty:
            advanced_data[chain] = df
            print(f"  ✓ {chain}: {len(df)} items extracted")

if advanced_data:
    print(f"\n✅ Extracted data for {len(advanced_data)} chains using advanced methods")
    # Add to public_data for merging
    if 'public_data' not in locals():
        public_data = {}
    public_data.update(advanced_data)


🔍 Trying advanced HTML extraction for missing chains...


## 6) Manual Data Collection & API Endpoints

For restaurants that use JavaScript to load data dynamically, you have these options:

1. **Inspect Network Calls**: Open browser DevTools (F12) → Network tab → Filter by XHR/Fetch → Look for JSON API endpoints
2. **Use Selenium/Playwright**: For JavaScript-heavy sites, use browser automation
3. **Manual Entry**: Use the cell below to add manually collected data

### Known API Endpoints (to be discovered):
- McDonald's: Check network calls on nutrition calculator page
- Starbucks: Menu data may be in API calls
- Subway: May have JSON endpoints for nutrition data
- Others: Inspect network calls in browser DevTools

In [12]:
# Manual data entry for chains that require JavaScript/API access
# Format: List of dictionaries with chain, item_name, and nutrition fields

manual_data = [
    # Example format:
    # {
    #     "chain": "McDonald's",
    #     "item_name": "Big Mac",
    #     "calories": 563,
    #     "fat": 33,
    #     "carbs": 43,
    #     "protein": 25,
    #     "sodium": 1010
    # },
    # Add your manually collected data here
]

# Add manual data to the dataset
if manual_data:
    print(f"Adding {len(manual_data)} manually entered items...")
    for item in manual_data:
        rows.append(standardize_row(item))
    print(f"Total rows after manual entry: {len(rows)}")
else:
    print("No manual data entered. Use this cell to add data from JavaScript-heavy sites.")

No manual data entered. Use this cell to add data from JavaScript-heavy sites.


In [13]:
# Create final dataset and show summary
df_final = pd.DataFrame(rows)

# Expected chains - 25+ chains covering diverse cuisines and cooking practices
# This diversity is important for AI training to understand different:
# - Cooking methods (fried, grilled, baked, steamed, stir-fried, wok, broiled, tandoor, flame-grilled)
# - Cuisines (American, Mexican, Italian, Asian, Mediterranean, Seafood, Steakhouse, Chinese, Indian, African)
# - Meal types (fast food, casual dining, coffee shops, pizza, breakfast, seafood, curry houses)
expected_chains = [
    # American Fast Food (fried, grilled)
    "McDonald's", "Burger King", "Wendy's", "Chick-fil-A", "Arby's", "Sonic", "Jack in the Box",
    # Premium Burgers (grilled, fresh)
    "Five Guys", "Shake Shack", "In-N-Out Burger",
    # Mexican/Tex-Mex (spices, beans, rice, grilled)
    "Taco Bell", "Chipotle", "Qdoba", "Moe's Southwest Grill",
    # Pizza/Italian (baked, dough-based)
    "Domino's", "Pizza Hut",
    # Italian-American Casual Dining (pasta, baked, grilled)
    "Olive Garden", "Fazoli's",
    # Coffee/Café (beverages, pastries)
    "Starbucks", "Dunkin'",
    # Sandwich/Deli (assembled, fresh)
    "Subway", "Panera Bread",
    # Asian (stir-fry, wok, steamed, fried)
    "KFC", "Panda Express",
    # Mediterranean (grilled, fresh, assembled)
    "Cava",
    # Breakfast/Diner (griddle, fried, baked)
    "IHOP", "Denny's",
    # Seafood (fried, grilled, steamed)
    "Red Lobster", "Long John Silver's",
    # Steakhouse (grilled, broiled)
    "Outback Steakhouse", "Texas Roadhouse",
    # Casual Dining (varied cooking methods)
    "Applebee's",
    # ===== ADDITIONAL DIVERSE CUISINES =====
    # Chinese-American (wok, stir-fry, Szechuan)
    "P.F. Chang's", "Pei Wei",
    # African/Portuguese (flame-grilled, peri-peri)
    "Nando's",
    # Indian (curry, tandoor, spices)
    "Curry House", "Bombay Express"
]

print("=" * 60)
print("📊 FINAL DATASET SUMMARY")
print("=" * 60)
print(f"Total items: {len(df_final)}")
print(f"Total columns: {len(df_final.columns)}")

if len(df_final) > 0:
    print(f"\nData by chain:")
    chain_counts = df_final['chain'].value_counts()
    print(chain_counts.to_string())
    
    # Normalize chain names to count unique chains (remove "(PDF)", "(HTML)" variations)
    unique_base_chains = set()
    chain_variations = {}
    for chain in df_final['chain'].unique():
        base_name = str(chain).split(' (')[0].strip()
        unique_base_chains.add(base_name)
        if base_name not in chain_variations:
            chain_variations[base_name] = []
        chain_variations[base_name].append(chain)
    
    print(f"\n📊 Unique Chains Collected: {len(unique_base_chains)}")
    print(f"   Chains: {', '.join(sorted(unique_base_chains))}")
    
    # Show which chains we have vs missing
    found_chains = set(df_final['chain'].unique())
    missing_chains = [c for c in expected_chains if c not in found_chains and not any(c.lower() in str(fc).lower() for fc in found_chains)]
    
    # Check if we have at least 15 unique chains
    if len(unique_base_chains) >= 15:
        print(f"\n✅ SUCCESS: {len(unique_base_chains)} unique chains collected (target: 15+)")
        print("   Dataset has sufficient diversity for AI training!")
    elif len(unique_base_chains) >= 10:
        print(f"\n⚠️  PARTIAL: {len(unique_base_chains)} unique chains collected (target: 15+)")
        print("   Consider adding more chains for better diversity")
    else:
        print(f"\n⚠️  INSUFFICIENT: Only {len(unique_base_chains)} unique chains collected (target: 15+)")
        print("   Dataset may lack sufficient diversity for comprehensive AI training")
    
    if missing_chains:
        print(f"\n⚠️  Missing chains ({len(missing_chains)}): {', '.join(missing_chains[:10])}")
        if len(missing_chains) > 10:
            print(f"   ... and {len(missing_chains) - 10} more")
        print("   → Use manual data entry cell (Cell 19) or find alternative data sources")
    
    # Filter out None values from column names
    column_names = [str(col) if col is not None else 'unnamed' for col in df_final.columns[:10]]
    print(f"\nColumns: {', '.join(column_names)}...")
    
    # Show sample rows - handle missing columns gracefully
    sample_cols = ['chain', 'item_name']
    for col in ['calories', 'fat', 'carbs', 'protein']:
        if col in df_final.columns:
            sample_cols.append(col)
    
    print(f"\nSample rows:")
    print(df_final[sample_cols].head(10).to_string())
else:
    print("\n⚠️  No data collected! Check download and processing steps.")

print("=" * 60)

📊 FINAL DATASET SUMMARY
Total items: 2559
Total columns: 134

Data by chain:
chain
Dunkin' (PDF)         1016
McDonald's             471
Domino's               421
Olive Garden (PDF)     328
Panera Bread           182
Subway (PDF)            95
Jack in the Box         13
Burger King             11
Sonic                   11
Chick-fil-A              5
Curry House              4
Pei Wei                  2

📊 Unique Chains Collected: 12
   Chains: Burger King, Chick-fil-A, Curry House, Domino's, Dunkin', Jack in the Box, McDonald's, Olive Garden, Panera Bread, Pei Wei, Sonic, Subway

⚠️  PARTIAL: 12 unique chains collected (target: 15+)
   Consider adding more chains for better diversity

⚠️  Missing chains (25): Wendy's, Arby's, Five Guys, Shake Shack, In-N-Out Burger, Taco Bell, Chipotle, Qdoba, Moe's Southwest Grill, Pizza Hut
   ... and 15 more
   → Use manual data entry cell (Cell 19) or find alternative data sources

Columns: chain, item_name, )lack(
seirolaC, unnamed, )g(
taF, ytta

In [14]:

# Save processed dataset in multiple formats
# Ensure df_final exists (created in previous cell)
if 'df_final' not in locals():
    df_final = pd.DataFrame(rows)

df_final.to_csv("data/processed/restaurant_nutrition_dataset.csv", index=False)
df_final.to_json("data/processed/restaurant_nutrition_dataset.jsonl", orient="records", lines=True)
df_final.to_json("data/processed/restaurant_nutrition_dataset.json", orient="records", indent=2)

print(f"\n✅ Dataset saved successfully!")
print(f"   📄 CSV: data/processed/restaurant_nutrition_dataset.csv")
print(f"   📄 JSONL: data/processed/restaurant_nutrition_dataset.jsonl")
print(f"   📄 JSON: data/processed/restaurant_nutrition_dataset.json")
print(f"   📊 Shape: {df_final.shape[0]} rows × {df_final.shape[1]} columns")

# Count unique chains (normalize variations like "(PDF)", "(HTML)")
unique_base_chains = set()
for chain in df_final['chain'].unique():
    base_name = str(chain).split(' (')[0].strip()
    unique_base_chains.add(base_name)

print(f"\n📋 Chains included ({len(unique_base_chains)} unique): {', '.join(sorted(unique_base_chains))}")

if len(unique_base_chains) >= 15:
    print(f"\n🎉 Dataset meets diversity requirement: {len(unique_base_chains)} chains from various cuisines!")
    print("   Ready for AI training with diverse cooking methods and cuisines.")
elif len(unique_base_chains) >= 10:
    print(f"\n⚠️  Dataset has {len(unique_base_chains)} chains (target: 15+)")
    print("   Consider adding more chains using manual data entry (Cell 19) for better diversity.")
else:
    print(f"\n⚠️  Dataset has only {len(unique_base_chains)} chains (target: 15+)")
    print("   Please add more chains using manual data entry (Cell 19) to meet diversity requirements.")



✅ Dataset saved successfully!
   📄 CSV: data/processed/restaurant_nutrition_dataset.csv
   📄 JSONL: data/processed/restaurant_nutrition_dataset.jsonl
   📄 JSON: data/processed/restaurant_nutrition_dataset.json
   📊 Shape: 2559 rows × 134 columns

📋 Chains included (12 unique): Burger King, Chick-fil-A, Curry House, Domino's, Dunkin', Jack in the Box, McDonald's, Olive Garden, Panera Bread, Pei Wei, Sonic, Subway

⚠️  Dataset has 12 chains (target: 15+)
   Consider adding more chains using manual data entry (Cell 19) for better diversity.


## 7) Train Layer 2 Calibration Model

This cell trains the Layer 2 calibration model on the collected restaurant data.

In [15]:
# Train Layer 2 Calibration Model
# NOTE: If you see pickle errors, restart the kernel (Kernel -> Restart Kernel)
#       then re-run this cell to pick up the latest code changes.

import sys
from pathlib import Path
import random
import os

# Add layer2 to path
sys.path.insert(0, str(Path.cwd()))

# Force reload of layer2 module to pick up any code changes
if 'layer2' in sys.modules:
    import importlib
    importlib.reload(sys.modules['layer2'])
    importlib.reload(sys.modules['layer2.calibration_model'])

# Try imports with fallback
try:
    from layer2 import CalibrationModel, set_model
    from layer2.schemas import BaselineEstimate, RestaurantTruth
except ImportError:
    # Fallback to direct imports
    try:
        from layer2.calibration_model import CalibrationModel
        from layer2.inference import set_model
        from layer2.schemas import BaselineEstimate, RestaurantTruth
    except ImportError as e:
        print(f"❌ Could not import Layer 2: {e}")
        print("   Please ensure layer2 package is in the current directory")
        raise

# Now proceed with training
print("=" * 60)
print("Training Layer 2 Calibration Model")
print("=" * 60)

try:
    
    # Prepare training data from df_final
    if 'df_final' not in locals() and 'df_final' not in globals():
        print("⚠️  df_final not found. Creating from rows...")
        if 'rows' in locals() or 'rows' in globals():
            df_final = pd.DataFrame(rows)
        else:
            print("❌ Neither df_final nor rows found. Please run previous cells first.")
            raise ValueError("No data available for training")
    
    # Find nutrition columns (handle various column name formats)
    nutrition_cols = {}
    for col in df_final.columns:
        col_lower = str(col).lower()
        if 'calorie' in col_lower or col_lower == 'calories':
            nutrition_cols['calories'] = col
        elif ('fat' in col_lower and 'trans' not in col_lower and 'saturated' not in col_lower) or col_lower == 'fat':
            nutrition_cols['fat'] = col
        elif 'carb' in col_lower or col_lower == 'carbs':
            nutrition_cols['carbs'] = col
        elif 'protein' in col_lower or col_lower == 'protein':
            nutrition_cols['protein'] = col
        elif 'sodium' in col_lower or col_lower == 'sodium':
            nutrition_cols['sodium'] = col
    
    print(f"\n📊 Found nutrition columns:")
    for macro, col in nutrition_cols.items():
        print(f"   {macro}: {col}")
    
    if not nutrition_cols:
        print("⚠️  No nutrition columns found! Using first numeric columns...")
        numeric_cols = df_final.select_dtypes(include=['float64', 'int64']).columns
        if len(numeric_cols) >= 5:
            nutrition_cols = {
                'calories': numeric_cols[0],
                'fat': numeric_cols[1],
                'carbs': numeric_cols[2],
                'protein': numeric_cols[3],
                'sodium': numeric_cols[4]
            }
    
    # Prepare training data
    baseline_estimates = []
    restaurant_truths = []
    restaurant_metadata = []
    
    valid_count = 0
    for idx, row in df_final.iterrows():
        try:
            # Get nutrition values - row is a pandas Series, use .get() or direct access
            calories_col = nutrition_cols.get('calories', 'calories')
            fat_col = nutrition_cols.get('fat', 'fat')
            carbs_col = nutrition_cols.get('carbs', 'carbs')
            protein_col = nutrition_cols.get('protein', 'protein')
            sodium_col = nutrition_cols.get('sodium', 'sodium')
            
            # Get values from pandas Series - use direct indexing with fallback
            def safe_get(row, col, default=0):
                try:
                    val = row[col] if col in row.index else default
                    return float(val) if pd.notna(val) else default
                except:
                    return default
            
            calories = safe_get(row, calories_col, 0)
            fat = safe_get(row, fat_col, 0)
            carbs = safe_get(row, carbs_col, 0)
            protein = safe_get(row, protein_col, 0)
            sodium = safe_get(row, sodium_col, 0)
            
            chain = str(row.get('chain', 'Unknown') if 'chain' in row.index else 'Unknown')
            item_name = str(row.get('item_name', 'Unknown') if 'item_name' in row.index else 'Unknown')
        except Exception as e:
            continue  # Skip rows that cause errors
        
        # Skip if no valid nutrition data or invalid values
        if calories <= 0 and fat <= 0 and carbs <= 0:
            continue
        
        # Additional validation - skip NaN values
        try:
            if pd.isna(calories) or pd.isna(fat) or pd.isna(carbs):
                continue
        except:
            # If pd.isna fails, check with numpy or simple comparison
            import math
            if math.isnan(calories) or math.isnan(fat) or math.isnan(carbs):
                continue
        
        # Create simulated baseline (Layer 1 would provide this)
        # Simulate 10-20% variation
        noise = {
            'calories': random.uniform(0.85, 1.15),
            'fat': random.uniform(0.80, 1.20),
            'carbs': random.uniform(0.85, 1.15),
            'protein': random.uniform(0.88, 1.12),
            'sodium': random.uniform(0.75, 1.25),
        }
        
        baseline = BaselineEstimate(
            item_name=item_name,
            ingredients=[],
            cooking_methods=["fried"],  # Would come from Layer 1
            sauces=[],
            portion_class="entree",
            macros={
                "calories": max(0, calories * noise['calories']),
                "fat": max(0, fat * noise['fat']),
                "carbs": max(0, carbs * noise['carbs']),
                "protein": max(0, protein * noise['protein']),
                "sodium": max(0, sodium * noise['sodium']),
            }
        )
        
        truth = RestaurantTruth(
            chain=chain,
            item_name=item_name,
            calories=calories,
            fat=fat,
            carbs=carbs,
            protein=protein,
            sodium=sodium,
        )
        
        # Only add if we have valid data
        if calories > 0 or fat > 0 or carbs > 0:
            baseline_estimates.append(baseline)
            restaurant_truths.append(truth)
            restaurant_metadata.append({"restaurant": chain})
            valid_count += 1
        
        if valid_count >= 2000:  # Limit for notebook execution
            break
    
    print(f"\n✅ Prepared {valid_count} training samples")
    
    if valid_count == 0:
        print("❌ No valid training samples found!")
        print("   Check that your dataset has nutrition data (calories, fat, carbs, etc.)")
        raise ValueError("No valid training data")
    
    # Train model
    print(f"\n🔧 Training calibration model...")
    model = CalibrationModel()
    model.train(baseline_estimates, restaurant_truths, restaurant_metadata)
    set_model(model)
    print(f"✅ Model trained successfully!")
    
    # Save model
    import pickle
    os.makedirs("layer2", exist_ok=True)
    try:
        with open("layer2/trained_model.pkl", "wb") as f:
            pickle.dump(model, f)
        print(f"✅ Model trained and saved to layer2/trained_model.pkl")
    except Exception as pickle_error:
        print(f"⚠️  Could not save model to pickle: {pickle_error}")
        print("   This might be due to cached code in the kernel.")
        print("   Try: Kernel -> Restart Kernel, then re-run this cell")
        print("   The model is still trained and available in memory (set_model was called)")
    
    # Print statistics
    print(f"\n📊 Model Statistics:")
    print(f"   Restaurants learned: {len(model.multipliers['restaurant'])}")
    print(f"   Cuisines learned: {len(model.multipliers['cuisine'])}")
    print(f"   Cooking methods learned: {len(model.multipliers['cooking_method'])}")
    
    # Show sample counts
    restaurant_counts = {}
    for restaurant, macros in model.sample_counts['restaurant'].items():
        total = sum(macros.values())
        if total > 0:
            restaurant_counts[restaurant] = total
    
    print(f"\n   Top 5 restaurants by sample count:")
    for restaurant, count in sorted(restaurant_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"     {restaurant}: {count} samples")
    
    print(f"\n✅ Layer 2 model is ready for use!")
    
except Exception as e:
    print(f"❌ Error training model: {e}")
    import traceback
    traceback.print_exc()
    print("\n💡 Troubleshooting:")
    print("   1. Make sure you've run previous cells to create df_final")
    print("   2. Check that your dataset has nutrition columns")
    print("   3. Verify layer2 package is in the current directory")


Training Layer 2 Calibration Model

📊 Found nutrition columns:
   protein: protein
   calories: Calories
from Fat
   fat: fat
   sodium: sodium
   carbs: carbs

✅ Prepared 709 training samples

🔧 Training calibration model...
✅ Model trained successfully!
✅ Model trained and saved to layer2/trained_model.pkl

📊 Model Statistics:
   Restaurants learned: 6
   Cuisines learned: 2
   Cooking methods learned: 1

   Top 5 restaurants by sample count:
     McDonald's: 1616 samples
     Olive Garden (PDF): 218 samples
     Jack in the Box: 39 samples
     Burger King: 33 samples
     Sonic: 33 samples

✅ Layer 2 model is ready for use!


## 7.5) Verify Kernel Restart (Run After Restarting Kernel)

**⚠️ IMPORTANT:** If you just restarted the kernel, run this cell first to verify the pickle fix is loaded.

## 8) Test Layer 2 Calibration

Test the calibration engine with a sample item.

In [16]:
# Test Layer 2 Calibration
try:
    # Try importing from package first
    try:
        from layer2 import calibrate
        from layer2.schemas import BaselineEstimate
    except ImportError:
        # Fallback to direct imports
        from layer2.inference import calibrate
        from layer2.schemas import BaselineEstimate
    
    import pickle
    
    # Load model if not already loaded
    try:
        with open("layer2/trained_model.pkl", "rb") as f:
            model = pickle.load(f)
        try:
            from layer2 import set_model
        except ImportError:
            from layer2.inference import set_model
        set_model(model)
        print("✅ Model loaded")
    except FileNotFoundError:
        print("⚠️  Model not found - using fallback mode")
        print("   Run Cell 24 first to train the model")
        model = None
    except Exception as e:
        print(f"⚠️  Could not load model: {e}")
        model = None
    
    # Test calibration with a sample
    print("\n" + "=" * 60)
    print("Testing Layer 2 Calibration")
    print("=" * 60)
    
    # Example: McDonald's Big Mac
    baseline = BaselineEstimate(
        item_name="Big Mac",
        ingredients=["beef patty", "bun", "cheese", "lettuce", "pickles"],
        cooking_methods=["fried", "grilled"],
        sauces=["mayo", "special sauce"],
        portion_class="entree",
        macros={
            "calories": 500.0,
            "fat": 25.0,
            "carbs": 45.0,
            "protein": 20.0,
            "sodium": 900.0
        }
    )
    
    result = calibrate(
        baseline_estimate=baseline,
        restaurant_metadata={"restaurant": "McDonald's"},
        model=model
    )
    
    print(f"\n📊 Calibration Results for: {baseline['item_name']}")
    print(f"   Restaurant: McDonald's")
    print(f"\n   Adjusted Macros:")
    for macro in ["calories", "fat", "carbs", "protein", "sodium"]:
        baseline_val = baseline["macros"][macro]
        adjusted_val = result["adjusted_macros"][macro]
        multiplier = result["applied_adjustments"][macro]["multiplier"]
        confidence = result["confidence"][macro]
        adjustment_type = result["applied_adjustments"][macro]["adjustment_type"]
        
        change = adjusted_val - baseline_val
        change_pct = (change / baseline_val * 100) if baseline_val > 0 else 0
        
        print(f"     {macro:10s}: {baseline_val:7.1f} → {adjusted_val:7.1f} "
              f"({change:+.1f}, {change_pct:+.1f}%) "
              f"[×{multiplier:.3f}, conf={confidence:.2f}, {adjustment_type}]")
    
    print(f"\n✅ Calibration test complete!")
    
except ImportError as e:
    print(f"⚠️  Could not import Layer 2: {e}")
    print("   Install dependencies: pip install numpy pandas")
except Exception as e:
    print(f"❌ Error testing calibration: {e}")
    import traceback
    traceback.print_exc()


✅ Model loaded

Testing Layer 2 Calibration

📊 Calibration Results for: Big Mac
   Restaurant: McDonald's

   Adjusted Macros:
     calories  :   500.0 →   500.0 (+0.0, +0.0%) [×1.000, conf=0.79, default]
     fat       :    25.0 →    25.2 (+0.2, +0.9%) [×1.009, conf=0.85, restaurant]
     carbs     :    45.0 →    45.1 (+0.1, +0.3%) [×1.003, conf=0.91, restaurant]
     protein   :    20.0 →    20.2 (+0.2, +1.0%) [×1.010, conf=0.91, restaurant]
     sodium    :   900.0 →   912.6 (+12.6, +1.4%) [×1.014, conf=0.85, restaurant]

✅ Calibration test complete!



## 6) Next steps (Layer 2 integration)

Once you have this dataset, you can:

1. Run Layer 1 baseline estimates for each `item_name`
2. Extract Layer 2 signals using your ontology extractor
3. Join baseline + restaurant macros + signals into a training set

---

### Interactive calculators (Chipotle, Applebee's, etc.)
These require **XHR inspection** to find JSON endpoints.  
Once you have the endpoint, add a fetcher under `src/fetchers/` and parse the JSON.

Example structure:

```
src/fetchers/chipotle.py
src/parsers/json_parser.py
```

---

If you want, I can generate the `src/` package code next and integrate Layer 2 directly into this same project.
